In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, '->', dirs, files[:3])


/kaggle/input -> ['datasets'] []
/kaggle/input/datasets -> ['ashiqurrahmansunny', 'hsankesara'] []
/kaggle/input/datasets/ashiqurrahmansunny -> ['image-caption-generator-code'] []
/kaggle/input/datasets/ashiqurrahmansunny/image-caption-generator-code -> ['image-caption-generator'] []
/kaggle/input/datasets/ashiqurrahmansunny/image-caption-generator-code/image-caption-generator -> ['model', 'templates', 'data'] ['train.py', 'README.md', 'train_on_colab.ipynb']
/kaggle/input/datasets/ashiqurrahmansunny/image-caption-generator-code/image-caption-generator/model -> [] ['caption_model.py']
/kaggle/input/datasets/ashiqurrahmansunny/image-caption-generator-code/image-caption-generator/templates -> [] ['index.html']
/kaggle/input/datasets/ashiqurrahmansunny/image-caption-generator-code/image-caption-generator/data -> [] ['dataset.py']
/kaggle/input/datasets/hsankesara -> ['flickr-image-dataset'] []
/kaggle/input/datasets/hsankesara/flickr-image-dataset -> ['flickr30k_images'] []
/kaggle/input/

In [2]:
!cp -r /kaggle/input/datasets/*/image-caption-generator-code/image-caption-generator /kaggle/working/
%cd /kaggle/working/image-caption-generator
!ls


/kaggle/working/image-caption-generator
app.py		     README.md	       train_on_colab.ipynb
data		     requirements.txt  train_on_kaggle.ipynb
generate_caption.py  streamlit_app.py  train.py
model		     templates


## Install dependencies

In [3]:
!pip install -q pandas tqdm


## Set the dataset paths

In [4]:
IMAGE_DIR = "/kaggle/input/datasets/hsankesara/flickr-image-dataset/flickr30k_images/flickr30k_images"
CAPTIONS_CSV = "/kaggle/input/datasets/hsankesara/flickr-image-dataset/flickr30k_images/results.csv"

import os
print(os.path.exists(IMAGE_DIR), os.path.exists(CAPTIONS_CSV))


True True


## Training

In [5]:
!python train.py --image_dir "$IMAGE_DIR" --captions_csv "$CAPTIONS_CSV" \
    --checkpoint_dir /kaggle/working/checkpoints \
    --epochs 20 --batch_size 32 --fine_tune_start_epoch 8 --early_stopping_patience 4


Using device: cuda
Vocabulary size: 7727
Epoch 1/20 [train]:   0%|                              | 0/4718 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
Epoch 1/20 [train]: 100%|████████| 4718/4718 [12:55<00:00,  6.08it/s, loss=4.05]
Epoch 1: train_loss=4.3080  val_loss=3.9739
  -> saved new best checkpoint
Epoch 2/20 [train]: 100%|█████████| 4718/4718 [12:58<00:00,  6.06it/s, loss=3.9]
Epoch 2: train_loss=3.9585  val_loss=3.8725
  -> saved new best checkpoint
Epoch 3/20 [train]: 100%|████████| 4718/4718 [13:01<00:00,  6.04it/s, loss=3.41]
Epoch 3: train_loss=3.8593  val_loss=3.8013
  -> saved new best checkpoint
Epoch 4/20 [train]: 100%|████████| 4718/4718 [12:57<00:00,  6.07it/s, loss=3.47]
Epoch 4: train_loss=3.7878  val_loss=3.7600
  -> saved new best checkpoint
Epoch 5/20 [train]: 100%

In [10]:
import glob
sample_image = glob.glob(f"{IMAGE_DIR}/*.jpg")[0]
!python generate_caption.py --image "$sample_image" \
    --checkpoint /kaggle/working/checkpoints/best.pt \
    --vocab /kaggle/working/checkpoints/vocab.pkl


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
Caption: a man in a blue shirt is balancing a tray of food on his head


## Download checkpoints

In [8]:
!zip -r /kaggle/working/checkpoints.zip /kaggle/working/checkpoints


  adding: kaggle/working/checkpoints/ (stored 0%)
  adding: kaggle/working/checkpoints/vocab.pkl (deflated 46%)
  adding: kaggle/working/checkpoints/last.pt (deflated 7%)
  adding: kaggle/working/checkpoints/best.pt (deflated 7%)
